# A-Agent LoRA Fine-Tuning: gpt-oss-20B on AMD MI300X

Fine-tune `unsloth/gpt-oss-20b-BF16` as the **Answer Agent** for the AAIPL competition.

- **Model**: gpt-oss-20B (~40GB BF16)
- **Hardware**: AMD MI300X (192GB HBM3)
- **Method**: LoRA SFT with `train_on_responses_only`
- **Dataset**: 127k MCQs (Syllogisms, Blood Relations, Seating, Series)
- **Output**: LoRA adapters + merged 16-bit model

## 1. Installation

Run this cell only if Unsloth is not already installed.

In [ ]:
# Skip if Unsloth is already installed on the server
# !pip install --upgrade unsloth trl transformers datasets

## 2. Load Model with Unsloth

Load gpt-oss-20B in BF16 (no quantization) — fits comfortably on MI300X (192GB).
If loading from a local path, change `model_name` to the local directory.

In [ ]:
from unsloth import FastLanguageModel
import torch

# ═══ Configuration ═══
max_seq_length = 2048   # A-Agent: max_new_tokens=512, but input can be long
dtype = torch.bfloat16  # BF16 for ROCm / MI300X
load_in_4bit = False    # Full precision — we have 192GB VRAM

# Change to local path if models are pre-downloaded
# model_name = "/path/to/models--unsloth--gpt-oss-20b-BF16/snapshots/..."
model_name = "unsloth/gpt-oss-20b-BF16"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    trust_remote_code=True,
)

print(f"Model loaded: {model_name}")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")

## 3. Add LoRA Adapters

Attach LoRA adapters to all attention + MLP layers. Only these small adapter weights are trained.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                        # LoRA rank (higher = more capacity)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,               # 2x rank for faster convergence
    lora_dropout=0,              # 0 is optimized by Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 4. Load & Prepare Dataset

Load the A-Agent ChatML training data. The format is:
```json
{"messages": [
  {"role": "system", "content": "You are a logical reasoning expert..."},
  {"role": "user", "content": "<question + choices>"},
  {"role": "assistant", "content": "{\"answer\": \"B\", \"reasoning\": \"...\"}"}
]}
```

In [ ]:
import json
from datasets import Dataset

# ═══ Load dataset ═══
DATA_DIR = "data/final"  # Adjust path as needed on remote server

with open(f"{DATA_DIR}/aagent_chatml_train.json") as f:
    train_data = json.load(f)
with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_data = json.load(f)

print(f"Train: {len(train_data):,} samples")
print(f"Val:   {len(val_data):,} samples")
print(f"Sample keys: {list(train_data[0].keys())}")
print(f"\nSample messages[0]:")
for msg in train_data[0]['messages']:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")

In [ ]:
# Convert to HuggingFace Datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")

## 5. Apply Chat Template & Format Data

Use `standardize_sharegpt` to normalize the dataset format, then apply the gpt-oss chat template to produce the final training text.

In [ ]:
from unsloth.chat_templates import standardize_sharegpt

# Standardize message format (handles role name normalization)
train_dataset = standardize_sharegpt(train_dataset)
val_dataset = standardize_sharegpt(val_dataset)

# Formatting function: apply model's native chat template
def formatting_prompts_func(examples):
    # standardize_sharegpt may rename 'messages' to 'conversations'
    key = "conversations" if "conversations" in examples else "messages"
    convos = examples[key]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

# Preview formatted text
print("═" * 60)
print("Sample formatted training text:")
print("═" * 60)
print(train_dataset["text"][0][:500])
print("...")

## 6. Configure Training

Key settings for MI300X:
- **Batch size 16** with gradient accumulation 4 → effective batch 64
- **BF16** precision (native for MI300X)
- **`train_on_responses_only`** — loss computed only on assistant turns
- **1-3 epochs** over 114k training examples

### gpt-oss special tokens for `train_on_responses_only`:
- User turns start with: `<|start|>user<|message|>`
- Assistant turns start with: `<|start|>assistant<|channel|>final<|message|>`

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from transformers import DataCollatorForSeq2Seq

# ═══ Training Configuration ═══
OUTPUT_DIR = "aagent_gptoss_outputs"
NUM_EPOCHS = 1        # Start with 1, increase if val loss still dropping
BATCH_SIZE = 8        # MI300X can handle more; tune based on seq_length
GRAD_ACCUM = 8        # Effective batch = 8 * 8 = 64
LEARNING_RATE = 2e-4

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    packing=False,  # Don't pack — preserve conversation boundaries
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=50,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        bf16=True,
        dataloader_pin_memory=False,
        gradient_checkpointing=True,
        dataloader_num_workers=0,  # For ROCm stability
        remove_unused_columns=True,
    ),
)

# Train ONLY on assistant responses — mask system + user tokens from loss
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start|>user<|message|>",
    response_part="<|start|>assistant<|channel|>final<|message|>",
)

print(f"Training config:")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Total train samples: {len(train_dataset):,}")
print(f"  Total val samples:   {len(val_dataset):,}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Steps/epoch: ~{len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")

## 7. Memory Check (Optional)

Verify GPU memory usage before training starts.

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_mem = round(gpu_stats.total_memory / 1024**3, 2)
print(f"GPU: {gpu_stats.name}")
print(f"Total VRAM: {total_mem} GB")
print(f"Reserved before training: {reserved} GB")
print(f"Available for KV cache + training: {total_mem - reserved:.1f} GB")

## 8. Train!

Expected time: ~1-2 hours for 1 epoch with 114k samples on MI300X.

In [ ]:
# Switch to training mode
FastLanguageModel.for_training(model)

# Train
trainer_stats = trainer.train()

# Print final stats
print(f"\nTraining completed!")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"  Final loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")
peak_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"  Peak VRAM: {peak_mem} GB / {total_mem} GB ({100*peak_mem/total_mem:.1f}%)")

## 9. Save Model

Save as:
1. **LoRA adapters** — lightweight, load on top of base model later
2. **Merged 16-bit** — standalone model ready for deployment with vLLM

In [ ]:
# Save LoRA adapters (small, ~100-300MB)
LORA_PATH = "aagent_gptoss_lora"
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"LoRA adapters saved to: {LORA_PATH}")

# Save merged 16-bit model (full size, for vLLM deployment)
MERGED_PATH = "aagent_gptoss_merged_16bit"
print(f"Saving merged 16-bit model to: {MERGED_PATH} ...")
model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to: {MERGED_PATH}")

## 10. Quick Test — Inference

Test the fine-tuned A-Agent on a sample question from each category.

In [ ]:
# Switch to inference mode (2x faster)
FastLanguageModel.for_inference(model)

# Test questions — one per category
test_questions = [
    # Syllogism
    """Statements:
1. All dogs are animals.
2. All animals are living beings.
Conclusions:
I. All dogs are living beings.
II. Some living beings are dogs.

A) Only conclusion I follows
B) Only conclusion II follows
C) Both conclusions I and II follow
D) Neither conclusion I nor II follows""",

    # Blood Relations
    """A is the father of B. B is the mother of C. D is the brother of A. What is D to C?

A) Grandfather
B) Uncle
C) Great Uncle
D) Father""",

    # Seating Arrangement
    """5 persons A, B, C, D, E are seated in a row (left to right). 
B is to the immediate right of A. 
D is at one of the ends. 
C is to the immediate left of E.
Who is in the middle?

A) A
B) B
C) C
D) E""",

    # Series
    """Find the next term in the series: 2, 6, 18, 54, ?

A) 108
B) 162
C) 180
D) 216""",
]

system_prompt = (
    "You are a logical reasoning expert. Answer the given multiple-choice question.\n"
    'Provide your answer as a JSON object with "answer" (letter A/B/C/D) and "reasoning" (your step-by-step reasoning).'
)

for i, q in enumerate(test_questions):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    print(f"\n{'═'*50}")
    print(f"Q{i+1}: {q[:80]}...")
    print(f"A{i+1}: {response[:300]}")

## 11. Evaluate on Validation Set (Optional)

Run evaluation to check accuracy on held-out questions.

In [ ]:
import json
import random

# Quick accuracy check on a random subset of validation data
NUM_EVAL = 200  # Number of val samples to test
random.seed(42)

with open(f"{DATA_DIR}/aagent_chatml_val.json") as f:
    val_raw = json.load(f)

sample = random.sample(val_raw, min(NUM_EVAL, len(val_raw)))
correct = 0
total = 0

FastLanguageModel.for_inference(model)

for item in sample:
    msgs = item["messages"]
    # Extract ground truth from assistant response
    gt_response = msgs[-1]["content"]
    try:
        gt_answer = json.loads(gt_response)["answer"]
    except:
        continue

    # Build prompt (system + user only)
    prompt_msgs = msgs[:-1]  # Everything except assistant
    inputs = tokenizer.apply_chat_template(
        prompt_msgs,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )

    # Try to extract answer letter
    try:
        pred = json.loads(response)["answer"]
    except:
        # Fallback: look for A/B/C/D in response
        pred = None
        for letter in ["A", "B", "C", "D"]:
            if f'"answer": "{letter}"' in response or f'"answer":"{letter}"' in response:
                pred = letter
                break
        if pred is None:
            for letter in ["A", "B", "C", "D"]:
                if letter in response[:10]:
                    pred = letter
                    break

    if pred:
        total += 1
        if pred == gt_answer:
            correct += 1

    if total % 50 == 0 and total > 0:
        print(f"  Progress: {total}/{NUM_EVAL} evaluated, accuracy: {100*correct/total:.1f}%")

print(f"\nFinal Accuracy: {correct}/{total} = {100*correct/total:.1f}%")

## 12. Integration with AAIPL Skeleton

To use this fine-tuned model as the A-Agent in the competition:

1. Copy `aagent_gptoss_lora/` to the `hf_models/` directory
2. Update `agents/answer_model.py` to load via Unsloth:

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="hf_models/aagent_gptoss_lora",
    max_seq_length=2048,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(model)
```

Or use the merged 16-bit model directly with transformers/vLLM.

---

**Done!** The A-Agent is fine-tuned and ready for deployment.